# 🛠️ Telco Customer Churn Feature Engineering

## 🎯 Objective
This notebook transforms the EDA findings into a reproducible preprocessing workflow for modeling.

The goals are:

- apply consistent data cleaning rules
- define feature and target sets
- separate numeric and categorical variables
- build a reusable preprocessing pipeline
- prepare train/validation datasets for modeling

---

## 📌 Preprocessing Rules from EDA
- `customerID` is excluded from modeling
- `Churn` is converted to `Churn_binary`
- `SeniorCitizen` is treated as a categorical/binary feature
- `TotalCharges` is converted to numeric
- missing values from invalid `TotalCharges` entries are removed
- preprocessing must remain consistent between training and inference

In [35]:
# 2. Import libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import warnings

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

In [36]:
# 3. Load raw data

data_path = Path("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = pd.read_csv(data_path)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Apply the Same Core Cleaning Rules from EDA

We first apply the same preprocessing decisions used in `01_eda.ipynb`
so that feature engineering is consistent with the earlier analysis.

In [37]:
# 5. Convert TotalCharges to numeric

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [38]:
# 6. Drop rows with missing values caused by invalid TotalCharges

df = df.dropna().copy()

In [39]:
# 7. Create binary target variable

df["Churn_binary"] = df["Churn"].map({"No": 0, "Yes": 1})

In [40]:
# 8. Verify cleaned dataset

print("Shape after cleaning:", df.shape)
print("Total missing values:", df.isnull().sum().sum())
print("\nMissing values by column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

df[["TotalCharges", "Churn", "Churn_binary"]].head()

Shape after cleaning: (7032, 22)
Total missing values: 0

Missing values by column:
Series([], dtype: int64)


,TotalCharges,Churn,Churn_binary
0,29.85,No,0
1,1889.50,No,0
2,108.15,Yes,1
3,1840.75,No,0
4,151.65,Yes,1


### Interpretation
- The cleaned dataset now follows the same rules defined in EDA.
- `TotalCharges` is numeric.
- `Churn_binary` is ready for modeling.

## 2. Define Feature Set and Target

We now define:
- `X`: modeling features
- `y`: binary target

We also explicitly exclude:
- `customerID` (identifier)
- `Churn` (original text target)

In [41]:
# 11. Define target and excluded columns

target_col = "Churn_binary"
id_col = "customerID"
original_target_col = "Churn"

exclude_cols = [id_col, original_target_col, target_col]
feature_cols = [col for col in df.columns if col not in exclude_cols]

X = df[feature_cols].copy()
y = df[target_col].copy()

print("Number of features:", len(feature_cols))
print("Feature columns:", feature_cols)

Number of features: 19
Feature columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


In [42]:
# 12. Check X and y shapes

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target distribution:\n", y.value_counts(normalize=True))

X shape: (7032, 19)
y shape: (7032,)
Target distribution:
 Churn_binary
0    0.734215
1    0.265785
Name: proportion, dtype: float64


### Interpretation
- `X` contains only modeling features.
- `y` is the binary churn target.
- The class imbalance remains and should be considered during modeling.

## 3. Separate Numeric and Categorical Features

We preserve the EDA rule that:
- `SeniorCitizen` should be treated as categorical

In [43]:
# 15. Separate numeric and categorical columns

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

if "SeniorCitizen" in numeric_features:
    numeric_features.remove("SeniorCitizen")

if "SeniorCitizen" not in categorical_features:
    categorical_features.append("SeniorCitizen")

numeric_features = sorted(numeric_features)
categorical_features = sorted(categorical_features)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['MonthlyCharges', 'TotalCharges', 'tenure']
Categorical features: ['Contract', 'Dependents', 'DeviceProtection', 'InternetService', 'MultipleLines', 'OnlineBackup', 'OnlineSecurity', 'PaperlessBilling', 'Partner', 'PaymentMethod', 'PhoneService', 'SeniorCitizen', 'StreamingMovies', 'StreamingTV', 'TechSupport', 'gender']


In [44]:
# 16. Check feature counts

print("Number of numeric features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

Number of numeric features: 3
Number of categorical features: 16


In [45]:
# 16-1. Preview feature schema

feature_schema = {
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "target": target_col
}

feature_schema

{'numeric_features': ['MonthlyCharges', 'TotalCharges', 'tenure'],
 'categorical_features': ['Contract',
  'Dependents',
  'DeviceProtection',
  'InternetService',
  'MultipleLines',
  'OnlineBackup',
  'OnlineSecurity',
  'PaperlessBilling',
  'Partner',
  'PaymentMethod',
  'PhoneService',
  'SeniorCitizen',
  'StreamingMovies',
  'StreamingTV',
  'TechSupport',
  'gender'],
 'target': 'Churn_binary'}

### Interpretation
- Numeric and categorical variables are now explicitly separated.
- `SeniorCitizen` is treated as a binary categorical feature for encoding.

## 4. Train / Validation Split

We split the dataset before fitting preprocessing steps.
This prevents data leakage.

In [46]:
# 19. Split train and validation sets

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("y_train distribution:\n", y_train.value_counts(normalize=True))
print("y_valid distribution:\n", y_valid.value_counts(normalize=True))

X_train shape: (5625, 19)
X_valid shape: (1407, 19)
y_train distribution:
 Churn_binary
0    0.734222
1    0.265778
Name: proportion, dtype: float64
y_valid distribution:
 Churn_binary
0    0.734186
1    0.265814
Name: proportion, dtype: float64


### Interpretation
- Stratified splitting preserves the churn ratio in both train and validation sets.
- This makes later model evaluation more stable and fair.

## 5. Build Preprocessing Pipelines

We create:
- a numeric preprocessing pipeline
- a categorical preprocessing pipeline
- a combined `ColumnTransformer`

This structure can later be reused in training, inference, and deployment.

In [47]:
# 22. Build numeric preprocessing pipeline

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

numeric_transformer

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

In [48]:
# 23. Build categorical preprocessing pipeline

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

categorical_transformer

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))])

In [49]:
# 24. Combine preprocessing pipelines with ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['MonthlyCharges', 'TotalCharges', 'tenure']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['Contract', 'Dependents', 'DeviceProtection',
                                  'InternetService', 'MultipleLines',
                                  'OnlineBackup', 'OnlineSecurity',
                                  'PaperlessBilling', 'Partner',
                                  'PaymentMethod', 'PhoneService',
                                  'SeniorCitizen', 'StreamingMovies',
                                  'StreamingTV', 'TechSupport', 'gender'])])

### Interpretation
- Numeric features are imputed and scaled.
- Categorical features are imputed and one-hot encoded.
- `handle_unknown="ignore"` is important for inference robustness.

## 6. Fit Preprocessor on Training Data Only

We now fit the preprocessor on `X_train` only,
then transform both training and validation sets.

In [50]:
# 27. Fit preprocessor on training data

X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)

print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_valid shape:", X_valid_processed.shape)

# Convert sparse matrix to dense if needed
if hasattr(X_train_processed, "toarray"):
    X_train_processed = X_train_processed.toarray()

if hasattr(X_valid_processed, "toarray"):
    X_valid_processed = X_valid_processed.toarray()

Processed X_train shape: (5625, 46)
Processed X_valid shape: (1407, 46)


In [51]:
# 28. Inspect processed data types

print("Type of X_train_processed:", type(X_train_processed))
print("Type of X_valid_processed:", type(X_valid_processed))
print("Train dtype:", getattr(X_train_processed, "dtype", "mixed"))
print("Valid dtype:", getattr(X_valid_processed, "dtype", "mixed"))

Type of X_train_processed: <class 'numpy.ndarray'>
Type of X_valid_processed: <class 'numpy.ndarray'>
Train dtype: float64
Valid dtype: float64


### Interpretation
- The processed datasets are now ready for model training.
- Preprocessing has been fit only on training data, which avoids leakage.

## 7. Inspect Output Feature Names

After one-hot encoding, the number of final features increases.
We inspect the transformed feature names to understand the final modeling space.

In [52]:
# 31. Extract transformed feature names

all_feature_names = preprocessor.get_feature_names_out()

print("Total transformed features:", len(all_feature_names))
all_feature_names[:20]

Total transformed features: 46


array(['num__MonthlyCharges', 'num__TotalCharges', 'num__tenure',
       'cat__Contract_Month-to-month', 'cat__Contract_One year',
       'cat__Contract_Two year', 'cat__Dependents_No',
       'cat__Dependents_Yes', 'cat__DeviceProtection_No',
       'cat__DeviceProtection_No internet service',
       'cat__DeviceProtection_Yes', 'cat__InternetService_DSL',
       'cat__InternetService_Fiber optic', 'cat__InternetService_No',
       'cat__MultipleLines_No', 'cat__MultipleLines_No phone service',
       'cat__MultipleLines_Yes', 'cat__OnlineBackup_No',
       'cat__OnlineBackup_No internet service', 'cat__OnlineBackup_Yes'],
      dtype=object)

In [53]:
# 31-1. Validate transformed feature count

print("Expected transformed feature count:", len(all_feature_names))
print("Actual transformed train columns:", X_train_processed.shape[1])
print("Actual transformed valid columns:", X_valid_processed.shape[1])

Expected transformed feature count: 46
Actual transformed train columns: 46
Actual transformed valid columns: 46


In [54]:
# 32. Preview transformed feature names

all_feature_names

array(['num__MonthlyCharges', 'num__TotalCharges', 'num__tenure',
       'cat__Contract_Month-to-month', 'cat__Contract_One year',
       'cat__Contract_Two year', 'cat__Dependents_No',
       'cat__Dependents_Yes', 'cat__DeviceProtection_No',
       'cat__DeviceProtection_No internet service',
       'cat__DeviceProtection_Yes', 'cat__InternetService_DSL',
       'cat__InternetService_Fiber optic', 'cat__InternetService_No',
       'cat__MultipleLines_No', 'cat__MultipleLines_No phone service',
       'cat__MultipleLines_Yes', 'cat__OnlineBackup_No',
       'cat__OnlineBackup_No internet service', 'cat__OnlineBackup_Yes',
       'cat__OnlineSecurity_No',
       'cat__OnlineSecurity_No internet service',
       'cat__OnlineSecurity_Yes', 'cat__PaperlessBilling_No',
       'cat__PaperlessBilling_Yes', 'cat__Partner_No', 'cat__Partner_Yes',
       'cat__PaymentMethod_Bank transfer (automatic)',
       'cat__PaymentMethod_Credit card (automatic)',
       'cat__PaymentMethod_Electronic che

### Interpretation
- One-hot encoding expands categorical features into multiple binary columns.
- These transformed feature names should be tracked for interpretability and inference consistency.

## 8. Convert Processed Data to DataFrame (Optional for Inspection)

This step is useful for:
- debugging
- feature inspection
- verifying preprocessing outputs

In [55]:
# 35. Convert processed train data to DataFrame

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=all_feature_names,
    index=X_train.index
)

X_train_processed_df.head()

,num__MonthlyCharges,num__TotalCharges,num__tenure,cat__Contract_Month-to-month,cat__Contract_One year,cat__Contract_Two year,cat__Dependents_No,cat__Dependents_Yes,cat__DeviceProtection_No,cat__DeviceProtection_No internet service,...,cat__StreamingMovies_No internet service,cat__StreamingMovies_Yes,cat__StreamingTV_No,cat__StreamingTV_No internet service,cat__StreamingTV_Yes,cat__TechSupport_No,cat__TechSupport_No internet service,cat__TechSupport_Yes,cat__gender_Female,cat__gender_Male
1413,0.981556,1.659900,1.321816,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
7003,-0.971546,-0.562252,-0.267410,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
3355,0.837066,1.756104,1.444064,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
4494,0.641092,-0.908326,-1.204646,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3541,-0.808787,-0.101561,0.669826,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0


In [56]:
# 36. Convert processed validation data to DataFrame

X_valid_processed_df = pd.DataFrame(
    X_valid_processed,
    columns=all_feature_names,
    index=X_valid.index
)

X_valid_processed_df.head()

,num__MonthlyCharges,num__TotalCharges,num__tenure,cat__Contract_Month-to-month,cat__Contract_One year,cat__Contract_Two year,cat__Dependents_No,cat__Dependents_Yes,cat__DeviceProtection_No,cat__DeviceProtection_No internet service,...,cat__StreamingMovies_No internet service,cat__StreamingMovies_Yes,cat__StreamingTV_No,cat__StreamingTV_No internet service,cat__StreamingTV_Yes,cat__TechSupport_No,cat__TechSupport_No internet service,cat__TechSupport_Yes,cat__gender_Female,cat__gender_Male
974,0.363738,0.984674,1.077320,0.0,0.0,1.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
619,0.450100,-0.781798,-1.041649,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4289,-1.491376,-0.537223,0.873573,0.0,0.0,1.0,1.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3721,-1.473107,-0.994619,-1.245396,1.0,0.0,0.0,1.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
4533,1.333645,2.308692,1.566313,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0


In [57]:
# 36-1. Check processed data quality

print("Nulls in processed train data:", X_train_processed_df.isnull().sum().sum())
print("Nulls in processed valid data:", X_valid_processed_df.isnull().sum().sum())

Nulls in processed train data: 0
Nulls in processed valid data: 0


### Interpretation
- The processed DataFrames make it easier to verify transformed values.
- This is helpful before moving into model experiments.

## 9. Final Summary

This notebook established a reproducible preprocessing workflow for churn modeling.

## 📌 Key Outputs

### 1. Cleaning Rules Applied
- `TotalCharges` converted to numeric
- rows with invalid `TotalCharges` removed
- `Churn` converted to `Churn_binary`

### 2. Feature Definition
- `customerID` excluded as identifier
- `Churn_binary` defined as target
- `SeniorCitizen` treated as categorical

### 3. Reproducible Preprocessing
- numeric pipeline: median imputation + scaling
- categorical pipeline: most frequent imputation + one-hot encoding
- `ColumnTransformer` used to unify preprocessing

### 4. Data Leakage Prevention
- train/validation split performed before preprocessing fit
- preprocessor fit only on training data

### 5. Next Step
- use the processed data in `03_model_experiments.ipynb`
- compare baseline models such as Logistic Regression and Random Forest
- evaluate using ROC-AUC, F1, Recall, and Precision